# Potencial Talents

## Importing Dependencies

In [1]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report, average_precision_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.compose import make_column_transformer

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

print('Modules Imported')

Modules Imported


## Import Data

In [2]:
data_path = 'potential-talents.xlsx'

In [3]:
data = pd.read_excel(data_path)

# Printing first values
data.head()

,id,job_title,location,connection,fit
0,1,2019 C.T. Bauer College of Business Graduate (...,"Houston, Texas",85,NaN
1,2,Native English Teacher at EPIK (English Progra...,Kanada,500+,NaN
2,3,Aspiring Human Resources Professional,"Raleigh-Durham, North Carolina Area",44,NaN
3,4,People Development Coordinator at Ryan,"Denton, Texas",500+,NaN
4,5,Advisory Board Member at Celal Bayar University,"İzmir, Türkiye",500+,NaN


## Exploring Dataset

**X:**

- `id`: unique identifier for candidate (numeric)
- `job_title`: job title for candidate (text)
- `location`: geographical location for candidate (text)
- `connections`: number of connections candidate has, 500+ means over 500 (text)

**Y:**

- `fit`: how fit the candidate is for the role? (numeric, probability between 0-1)

## Defining keywords

In [4]:
keywords = ['Aspiring human resources','seeking human resources'] 

These are the keywords that will be used to rank potential candidates

## Vector Search

In [5]:
job_titles = data['job_title']

job_titles.head()

0    2019 C.T. Bauer College of Business Graduate (...
1    Native English Teacher at EPIK (English Progra...
2                Aspiring Human Resources Professional
3               People Development Coordinator at Ryan
4      Advisory Board Member at Celal Bayar University
Name: job_title, dtype: object

Fetching job titles from the dataframe

In [6]:
# Creating the vectorize object
vectorizer = TfidfVectorizer()

# Creating vector based on the vectorization of the job titles feature
job_titles_matrix = vectorizer.fit_transform(job_titles)

# Creating vector based on the vectorization of the keywords list
keywords_matrix = vectorizer.transform(keywords)

In [7]:
# Geting a matrix with the similarity score from comparing the job titles and the keywords
resulting_matrix = cosine_similarity(job_titles_matrix, keywords_matrix)

# Getting the max score for all tokens from each candidate
resulting_matrix = resulting_matrix.max(axis=1)

In [8]:
# Adding the score data to the fit column
data = data.assign(fit=resulting_matrix)

data.head()

,id,job_title,location,connection,fit
0,1,2019 C.T. Bauer College of Business Graduate (...,"Houston, Texas",85,0.265312
1,2,Native English Teacher at EPIK (English Progra...,Kanada,500+,0.000000
2,3,Aspiring Human Resources Professional,"Raleigh-Durham, North Carolina Area",44,0.753591
3,4,People Development Coordinator at Ryan,"Denton, Texas",500+,0.000000
4,5,Advisory Board Member at Celal Bayar University,"İzmir, Türkiye",500+,0.000000


In [9]:
# Sorting candidates based on the "fit" feature (measures the similarity between the job title and the chosen keywords)
sorted_fit_data = data.sort_values(by=['fit'], ascending=False)

# Printing the top 10 candidates
sorted_fit_data.head(10)

,id,job_title,location,connection,fit
2,3,Aspiring Human Resources Professional,"Raleigh-Durham, North Carolina Area",44,0.753591
32,33,Aspiring Human Resources Professional,"Raleigh-Durham, North Carolina Area",44,0.753591
16,17,Aspiring Human Resources Professional,"Raleigh-Durham, North Carolina Area",44,0.753591
20,21,Aspiring Human Resources Professional,"Raleigh-Durham, North Carolina Area",44,0.753591
57,58,Aspiring Human Resources Professional,"Raleigh-Durham, North Carolina Area",44,0.753591
96,97,Aspiring Human Resources Professional,"Kokomo, Indiana Area",71,0.753591
45,46,Aspiring Human Resources Professional,"Raleigh-Durham, North Carolina Area",44,0.753591
23,24,Aspiring Human Resources Specialist,Greater New York City Area,1,0.695679
59,60,Aspiring Human Resources Specialist,Greater New York City Area,1,0.695679
5,6,Aspiring Human Resources Specialist,Greater New York City Area,1,0.695679


## Transforming Features

In [10]:
# Selecting features that will be used on future comparisons
selected_features = sorted_fit_data[['location','connection','fit']].copy()

# Transforming "connection" to integer format
selected_features['connection'] = selected_features['connection'].map(lambda x: 500 if x == "500+ " else pd.to_numeric(x))

In [11]:
# Preparing encoder
ohe = OneHotEncoder(sparse_output=False).set_output(transform='pandas')

# Encoding "location" using "one hot encoding"
ohe_transformation = ohe.fit_transform(selected_features[['location']])

# Adding the new encoding columns to the dataframe and removing the old "location" column
selected_features = pd.concat([selected_features, ohe_transformation], axis=1).drop(columns='location')

In [12]:
# Normalizing the "connection" and "fit" columns (the numeric features)
preprocessor = make_column_transformer(
    (StandardScaler(), ['connection', 'fit']),          
    remainder='passthrough'                              
)

# Fiting the normalized selected features in the X variable
X = preprocessor.fit_transform(selected_features)

## Selecting "Role" Candidate

In [13]:
# Selecting candidate, in this case the 7th candidate
role_idx = 6  

# Reshaping vector for vector operation (it is now a 1 x features vector
role_vector = X[role_idx].reshape(1, -1)

# Printing the vector for the "role" candidate
selected_features.iloc[role_idx]

connection                                      44.000000
fit                                              0.753591
location_Amerika Birleşik Devletleri             0.000000
location_Atlanta, Georgia                        0.000000
location_Austin, Texas Area                      0.000000
location_Baltimore, Maryland                     0.000000
location_Baton Rouge, Louisiana Area             0.000000
location_Bridgewater, Massachusetts              0.000000
location_Cape Girardeau, Missouri                0.000000
location_Chattanooga, Tennessee Area             0.000000
location_Chicago, Illinois                       0.000000
location_Dallas/Fort Worth Area                  0.000000
location_Denton, Texas                           0.000000
location_Gaithersburg, Maryland                  0.000000
location_Greater Atlanta Area                    0.000000
location_Greater Boston Area                     0.000000
location_Greater Chicago Area                    0.000000
location_Great

## Sorting Dataframe by "Role" Candidate similarity

In [14]:
# Get the similarity between ALL candidates in the dataframe and the "role" candidate
selected_features['fit'] = cosine_similarity(X, role_vector).flatten()

# Sort the candidates by "fit" (similarity with "role" candidate)
selected_features = selected_features.sort_values('fit', ascending=False)

# Print the first 10 candidates
selected_features.head(10)

,connection,fit,location_Amerika Birleşik Devletleri,"location_Atlanta, Georgia","location_Austin, Texas Area","location_Baltimore, Maryland","location_Baton Rouge, Louisiana Area","location_Bridgewater, Massachusetts","location_Cape Girardeau, Missouri","location_Chattanooga, Tennessee Area",...,"location_Milpitas, California","location_Monroe, Louisiana Area","location_Myrtle Beach, South Carolina Area","location_New York, New York","location_Raleigh-Durham, North Carolina Area",location_San Francisco Bay Area,"location_San Jose, California","location_Torrance, California","location_Virginia Beach, Virginia","location_İzmir, Türkiye"
2,44,1.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
32,44,1.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
16,44,1.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
20,44,1.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
57,44,1.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
45,44,1.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
96,71,0.834543,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
23,1,0.825223,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
59,1,0.825223,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,1,0.825223,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
